# `cloture` Interactive Frontier Eval Walkthrough

Zero-dependency Rust engine (`pip install cloture`, `<1s` install) for:
1. **Correlated LLM-as-a-Judge Auditing & LOO Pruning** on real **AllenAI RewardBench (2,985 items)** and **ScalerLab JudgeBench (350 objective items: o1-preview, Claude-3.5-Sonnet, GPT-4o, Gemini-1.5, Llama-3.1, Skywork)**.
2. **Anytime-Valid Consensus Stopping (`AscdStopper` / `CiteStopper`)** vs. naive peeking (`0%` vs `28.5%` false fires).
3. **Paired Sequential `AgentRace`** (`80.4%` fewer evals on real RewardBench with `100%` winner accuracy).
4. **Online Anytime-Valid `ActivePPI++`** (`100%` anytime coverage removing `-12.4 pp` to `-20.6 pp` cheap-proxy bias).

- Repo: https://github.com/pgg898200-ship-it/cloture
- Zero-Install Browser WASM Playground: https://pgg898200-ship-it.github.io/cloture/playground.html

In [ ]:
# Install zero-dependency universal wheel from PyPI (~1.3 MB, <1 second)
!pip install -q cloture>=0.2.0
import cloture
print("cloture version:", cloture.__version__)

## 1. Real-World Audit #1: AllenAI RewardBench (2,985 items, 11 frontier models)

Loads the compact bit-packed public receipts directly from GitHub Pages (zero API keys needed) and runs `cloture.audit_panel` + greedy LOO pruning (`11 -> 3`).

In [ ]:
import base64, json, time, urllib.request

RB_URL = "https://pgg898200-ship-it.github.io/cloture/benchmarks/rewardbench_receipts.json"
rb = json.loads(urllib.request.urlopen(RB_URL, timeout=20).read().decode())
N = rb["n_items"]
names = [m["name"] for m in rb["models"]]
K = len(names)
packed = base64.b64decode(rb["playground_preset_b64"])
votes = [[(packed[(i * K + j) >> 3] >> ((i * K + j) & 7)) & 1 for j in range(K)] for i in range(N)]
gold = [1] * N

t0 = time.perf_counter()
rep = cloture.audit_panel(votes, gold, n_perms=2000, n_condorcet_sims=2000, n_boot=1000, seed=42)
ms = (time.perf_counter() - t0) * 1000
print(f"Audited {K} models x {N} real items in {ms:.1f} ms!")
print(f"Full 11-model panel: acc={rep.panel_accuracy*100:.2f}% | best_single={rep.best_single_accuracy*100:.2f}% | n_eff={rep.n_eff_kish:.2f}/{K} | Condorcet gap={rep.condorcet_gap*100:+.2f} pp (p={rep.permutation_p_value:.5f})")

# Greedy LOO pruning 11 -> 3
active = list(range(K))
while len(active) > 3:
    sub = [[votes[i][j] for j in active] for i in range(N)]
    r = cloture.audit_panel(sub, gold, n_perms=100, n_condorcet_sims=100, n_boot=50, seed=42)
    drop_pos = max(range(len(active)), key=lambda p: (r.loo_delta_acc[p], r.loo_delta_neff[p]))
    active.pop(drop_pos)

r3 = cloture.audit_panel([[votes[i][j] for j in active] for i in range(N)], gold, n_perms=500, n_condorcet_sims=500, n_boot=200, seed=42)
print(f"Pruned 3-model triad {[names[j] for j in active]}: acc={r3.panel_accuracy*100:.2f}% (+{(r3.panel_accuracy - rep.panel_accuracy)*100:.2f} pp at -72.7% API cost!)")

## 2. Real-World Audit #2: ScalerLab JudgeBench (350 objective items, 13 frontier models incl. o1-preview & Claude-3.5-Sonnet)

Includes OpenAI (`o1-preview`, `o1-mini`, `GPT-4o`, `GPT-4o-mini`), Anthropic (`Claude-3.5-Sonnet`, `Haiku`), Google (`Gemini-1.5-Pro/Flash`), Meta (`Llama-3.1-405B/70B/8B`), and `Skywork` RM/Critic — plus **5-fold strictly out-of-sample CV** proving `+18.33 pp` out-of-sample accuracy gain at `-76.9%` lower API cost.

In [ ]:
JB_URL = "https://pgg898200-ship-it.github.io/cloture/benchmarks/judgebench_receipts.json"
jb = json.loads(urllib.request.urlopen(JB_URL, timeout=20).read().decode())
p = jb["playground_preset"]
v_bytes = base64.b64decode(p["votes_2bit_b64"])
g_bytes = base64.b64decode(p["gold_1bit_b64"])
s_bytes = base64.b64decode(p["strata_2bit_b64"])
Nj, Kj = p["n_items"], p["k_judges"]
jb_names = p["model_names"]
jb_gold = [(g_bytes[i >> 3] >> (i & 7)) & 1 for i in range(Nj)]
jb_strata = [(s_bytes[i >> 2] >> ((i & 3) * 2)) & 3 for i in range(Nj)]
jb_votes = [[(v_bytes[(i * Kj + j) >> 2] >> (((i * Kj + j) & 3) * 2)) & 3 for j in range(Kj)] for i in range(Nj)]

rep_jb = cloture.audit_panel(jb_votes, jb_gold, strata=jb_strata, n_perms=2000, n_condorcet_sims=2000, n_boot=1000, seed=42)
print(f"Full 13-model JudgeBench panel: acc={rep_jb.panel_accuracy*100:.2f}% | best_single (o1-preview)={rep_jb.best_single_accuracy*100:.2f}% | n_eff={rep_jb.n_eff_kish:.2f}/13")
cv = jb["cv5_out_of_sample_swap"]
print(f"5-Fold Out-of-Sample CV (13 -> 3): {cv['mean_full_test_accuracy']*100:.2f}% -> {cv['mean_pruned_test_accuracy']*100:.2f}% ({cv['out_of_sample_gain_pp']:+.2f} pp at -76.9% cost)")

## 3. Anytime-Valid Consensus Stopping (`AscdStopper`) vs. Naive Peeking

Watch `AscdStopper` stop easy prompts in ~21 calls (vs Fixed-64) while holding **0.0% false certifications on an exact 50/50 tie**, where naive `p < 0.05` peeking misfires ~28% of the time.

In [ ]:
import math, random

rng = random.Random(42)
ascd_false, naive_false = 0, 0
for _ in range(100):
    stopper = cloture.AscdStopper(k=2, delta=0.05)
    c0 = c1 = 0
    n_fired = False
    for step in range(1, 129):
        p0 = min(0.95, max(0.05, 0.5 + rng.gauss(0, 0.15)))
        d = stopper.update([p0, 1.0 - p0])
        if d.certified:
            ascd_false += 1
            break
        if rng.random() < p0: c0 += 1
        else: c1 += 1
        if not n_fired and step >= 6 and abs(c0 - c1) / math.sqrt(step) > 1.96:
            n_fired = True
    naive_false += int(n_fired)
print(f"Exact 50/50 tie-null false certifications (100 trials, delta=5%): cloture AscdStopper = {ascd_false}% | Naive p<0.05 peeking = {naive_false}%")

## 4. Anytime-Valid `ActivePPI++` on Real RewardBench (`GPT-4o-mini` proxy -> `GPT-4o`, 10% audit)

Even though `GPT-4o-mini` underestimates `GPT-4o` (`-5.86 pp` on the 1-bit binarized preset below; `-12.41 pp` on raw tie-penalized scores in `rewardbench_receipts.json`), `cloture.ActivePPI` maintains `100%` anytime coverage across all `2,985` items with strong variance reduction.

In [ ]:
idx_mini = names.index("GPT-4o-mini")
idx_4o = names.index("GPT-4o (2024-08)")
y_4o = [float(votes[i][idx_4o]) for i in range(N)]
f_mini = [float(votes[i][idx_mini]) for i in range(N)]
true_acc = sum(y_4o) / N
proxy_acc = sum(f_mini) / N

rng = random.Random(7)
order = list(range(N))
rng.shuffle(order)
ppi = cloture.ActivePPI(alpha=0.05, target_audit_rate=0.10, min_audit_prob=0.05)
covered_all = True
last = None
for i in order:
    pi = ppi.suggest_prob(f_mini[i], 1.0)
    audited = rng.random() < pi
    last = ppi.observe(f_mini[i], audited, y_4o[i] if audited else 0.0, pi)
    if not (last.lower <= true_acc <= last.upper):
        covered_all = False
print(f"True GPT-4o={true_acc*100:.2f}% | Biased GPT-4o-mini={proxy_acc*100:.2f}% ({(proxy_acc-true_acc)*100:+.2f} pp bias)")
print(f"ActivePPI++ (10% audit): [{last.lower*100:.2f}%, {last.upper*100:.2f}%] | covered_all_2985_steps={covered_all} | var_reduction={last.variance_reduction:.2f}x")